In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 88
==================================================
Week: 13 of 24
Day: 88 of 168
Date: January 23, 2026
Topic: Stock Price Predictor - Optimization & Backtesting

Week 13 Progress:
✅ Day 85: Time series fundamentals & data collection (COMPLETE!)
✅ Day 86: Feature engineering & preprocessing (COMPLETE!)
✅ Day 87: LSTM model training (COMPLETE!)
🔄 Day 88: Model optimization & backtesting (TODAY!)
⬜ Day 89: Interactive dashboard (Streamlit)
⬜ Day 90: Advanced features & deployment
⬜ Day 91: Final testing & portfolio addition

Progress: 43% (3/7 days)

==================================================
🎯 Week 13 Project: Stock Price Predictor with LSTM
Building on Day 87's baseline model, today we optimize and backtest!

🎯 Today's Learning Objectives:
1. Hyperparameter Tuning
   - LSTM units (64, 128, 256)
   - Sequence lengths (30, 60, 90, 120)
   - Dropout rates (0.1, 0.2, 0.3)
   - Learning rates (0.0001, 0.001, 0.01)

2. Walk-Forward Validation
   - Simulate real trading scenario
   - Retrain on expanding window
   - Test on next unseen period
   - Calculate cumulative performance

3. Trading Strategy & Backtesting
   - Generate buy/sell signals
   - Calculate returns (daily, cumulative)
   - Portfolio metrics (Sharpe ratio, drawdown)
   - Compare vs buy-and-hold

4. Performance Analysis
   - Win rate and profit factor
   - Risk-adjusted returns
   - Equity curve visualization
   - Trade-by-trade analysis

📚 Today's Structure:
Part 1 (2.5h): Hyperparameter Tuning & Optimization
Part 2 (2h): Walk-Forward Validation
Part 3 (2h): Backtesting & Trading Simulation
Part 4 (1h): Performance Analysis & Summary

🎯 SUCCESS CRITERIA:
✅ Tune hyperparameters for best performance
✅ Implement walk-forward validation
✅ Build trading strategy from predictions
✅ Calculate portfolio returns and metrics
✅ Achieve Sharpe ratio > 1.0
✅ Beat buy-and-hold strategy
✅ Visualize equity curves
✅ Document best model configurations

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys

print("Installing required libraries...")
print("=" * 80)

!pip install scikit-learn -q
!pip install yfinance -q
!pip install keras-tuner -q  # For hyperparameter tuning

print("\n✅ Libraries installed!")
print("=" * 80)

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

# Core
import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# Yahoo Finance
import yfinance as yf

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Machine Learning
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Hyperparameter Tuning
import keras_tuner as kt

# Install pandas-ta after sklearn
print("\nInstalling pandas-ta...")
!pip install pandas-ta -q
import pandas_ta as ta

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['figure.dpi'] = 100

print("\n✅ All libraries imported!")
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras Tuner version: {kt.__version__}")
print(f"✅ GPU available: {tf.config.list_physical_devices('GPU')}")
print("=" * 80)

Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 11.0 MB/s eta 0:00:00

✅ Libraries installed!

📚 IMPORTING LIBRARIES

Installing pandas-ta...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incomp

In [2]:
# ==================================================
# LOAD & PREPARE DATA (FROM DAY 87)
# ==================================================

print("\n" + "=" * 80)
print("📂 LOADING & PREPARING DATA")
print("=" * 80)

print("\n⏱️ Running full data pipeline from Day 87...")

# ── STEP 1: Download Data ──────────────────────────

stocks = {
    'TSLA': 'Tesla Inc.',
    'NVDA': 'NVIDIA Corporation',
    'AAPL': 'Apple Inc.',
    'MSFT': 'Microsoft Corporation',
    'JNJ':  'Johnson & Johnson',
}

end_date   = datetime.now()
start_date = end_date - timedelta(days=5*365)

stock_data = {}
for ticker, name in stocks.items():
    df = yf.download(ticker, start=start_date, end=end_date,
                     progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    if len(df) > 0:
        df['Ticker'] = ticker
        stock_data[ticker] = df

print(f"✅ Step 1: Downloaded data for {len(stock_data)} stocks")

# ── STEP 2: Engineer Features ─────────────────────

for ticker, df in stock_data.items():
    # Moving averages
    df['SMA_20']  = ta.sma(df['Close'], length=20)
    df['SMA_50']  = ta.sma(df['Close'], length=50)
    df['EMA_20']  = ta.ema(df['Close'], length=20)

    # RSI
    df['RSI_14'] = ta.rsi(df['Close'], length=14)

    # MACD
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['MACD']        = macd['MACD_12_26_9']
    df['MACD_Signal'] = macd['MACDs_12_26_9']
    df['MACD_Hist']   = macd['MACDh_12_26_9']

    # Bollinger Bands
    bb = ta.bbands(df['Close'], length=20, std=2)
    bb_cols = bb.columns.tolist()
    df['BB_Upper']   = bb[[c for c in bb_cols if c.startswith('BBU')][0]]
    df['BB_Lower']   = bb[[c for c in bb_cols if c.startswith('BBL')][0]]
    df['BB_Width']   = bb[[c for c in bb_cols if c.startswith('BBB')][0]]
    df['BB_Percent'] = bb[[c for c in bb_cols if c.startswith('BBP')][0]]

    # ATR & Stochastic
    df['ATR_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=14, d=3)
    df['STOCH_K'] = stoch['STOCHk_14_3_3']
    df['STOCH_D'] = stoch['STOCHd_14_3_3']

    # Lag features
    for lag in [1, 2, 3, 5, 10, 20, 30]:
        df[f'Close_Lag{lag}'] = df['Close'].shift(lag)

    df['Daily_Return'] = df['Close'].pct_change() * 100
    for lag in [1, 2, 3, 5]:
        df[f'Return_Lag{lag}'] = df['Daily_Return'].shift(lag)
        df[f'Volume_Lag{lag}'] = df['Volume'].shift(lag)

    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'Rolling_Mean_{window}'] = df['Close'].rolling(window).mean()
        df[f'Rolling_Std_{window}']  = df['Close'].rolling(window).std()
        df[f'Rolling_Min_{window}']  = df['Close'].rolling(window).min()
        df[f'Rolling_Max_{window}']  = df['Close'].rolling(window).max()
        df[f'Rolling_Range_{window}'] = df[f'Rolling_Max_{window}'] - df[f'Rolling_Min_{window}']

    # Price-based features
    df['Daily_Range']    = df['High'] - df['Low']
    df['Price_Position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])
    df['Gap']            = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1) * 100
    df['Log_Return']     = np.log(df['Close'] / df['Close'].shift(1)) * 100

print(f"✅ Step 2: Features engineered")

# ── STEP 3: Define Features & Clean ───────────────

feature_columns = [
    'Close', 'Open', 'High', 'Low', 'Volume',
    'SMA_20', 'SMA_50', 'EMA_20',
    'RSI_14',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Percent',
    'ATR_14', 'STOCH_K', 'STOCH_D',
    'Close_Lag1', 'Close_Lag2', 'Close_Lag3', 'Close_Lag5',
    'Return_Lag1', 'Return_Lag2',
    'Rolling_Mean_7', 'Rolling_Mean_30',
    'Rolling_Std_7', 'Rolling_Std_30',
    'Daily_Return', 'Log_Return',
    'Daily_Range', 'Price_Position', 'Gap',
]

target_column = 'Close'
target_idx    = feature_columns.index(target_column)

stock_data_clean = {}
for ticker, df in stock_data.items():
    df_clean = df[feature_columns].dropna()
    stock_data_clean[ticker] = df_clean

print(f"✅ Step 3: Data cleaned. Features: {len(feature_columns)}")
print(f"\n✅ Data pipeline complete! Ready for optimization!")
print("=" * 80)


📂 LOADING & PREPARING DATA

⏱️ Running full data pipeline from Day 87...
✅ Step 1: Downloaded data for 5 stocks
✅ Step 2: Features engineered
✅ Step 3: Data cleaned. Features: 34

✅ Data pipeline complete! Ready for optimization!


In [3]:
print("\n" + "=" * 80)
print("⚙️ PART 1: HYPERPARAMETER TUNING & OPTIMIZATION")
print("=" * 80)


⚙️ PART 1: HYPERPARAMETER TUNING & OPTIMIZATION


In [4]:
# ==================================================
# EXERCISE 1.1: HYPERPARAMETER TUNING THEORY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Hyperparameter Tuning Concepts")
print("=" * 80)

"""
📖 THEORY: Hyperparameter Tuning

Hyperparameters are settings we choose BEFORE training:
- Cannot be learned from data
- Require experimentation to find best values
- Huge impact on model performance

Key Hyperparameters for LSTM:

1. LSTM Units (64, 128, 256):
   - More units = more memory capacity
   - Too few = underfitting (can't learn patterns)
   - Too many = overfitting + slow training

2. Sequence Length (30, 60, 90, 120 days):
   - Longer = more historical context
   - Too short = missing important patterns
   - Too long = too much noise

3. Dropout Rate (0.1, 0.2, 0.3):
   - Prevents overfitting
   - Too low = overfitting
   - Too high = underfitting

4. Learning Rate (0.0001, 0.001, 0.01):
   - Controls how fast model learns
   - Too low = very slow convergence
   - Too high = unstable, won't converge

5. Number of LSTM Layers (1, 2, 3):
   - More layers = hierarchical learning
   - Too few = can't learn complex patterns
   - Too many = overfitting + slow

Tuning Methods:
- Grid Search: Try all combinations (slow but thorough)
- Random Search: Try random combinations (faster)
- Bayesian Optimization: Smart search (best!)

We'll use Keras Tuner with Bayesian Optimization!
"""

print("\n📊 Hyperparameters We'll Tune:")
print("   1. LSTM Units (Layer 1): 64, 128, 256")
print("   2. LSTM Units (Layer 2): 32, 64, 128")
print("   3. LSTM Units (Layer 3): 16, 32, 64")
print("   4. Dropout Rate: 0.1, 0.2, 0.3")
print("   5. Learning Rate: 0.0001, 0.001, 0.01")

print("\n⚙️ Tuning Strategy:")
print("   • Bayesian Optimization (smart search)")
print("   • 20 trials (balance speed vs thoroughness)")
print("   • Objective: Minimize validation loss")
print("   • Early stopping during each trial")

print("\n⏱️ Estimated Time:")
print("   • ~2-3 minutes per trial")
print("   • 20 trials = 40-60 minutes total")
print("   • GPU acceleration makes it feasible!")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Hyperparameter Tuning Concepts

📊 Hyperparameters We'll Tune:
   1. LSTM Units (Layer 1): 64, 128, 256
   2. LSTM Units (Layer 2): 32, 64, 128
   3. LSTM Units (Layer 3): 16, 32, 64
   4. Dropout Rate: 0.1, 0.2, 0.3
   5. Learning Rate: 0.0001, 0.001, 0.01

⚙️ Tuning Strategy:
   • Bayesian Optimization (smart search)
   • 20 trials (balance speed vs thoroughness)
   • Objective: Minimize validation loss
   • Early stopping during each trial

⏱️ Estimated Time:
   • ~2-3 minutes per trial
   • 20 trials = 40-60 minutes total
   • GPU acceleration makes it feasible!

✅ Exercise 1.1 Complete!


In [5]:
# ==================================================
# EXERCISE 1.2: BUILD TUNABLE LSTM MODEL
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Building Tunable LSTM Model")
print("=" * 80)

"""
📖 THEORY: Keras Tuner Model Builder

We create a model builder function that:
1. Takes a 'hp' (hyperparameter) object
2. Defines hyperparameter search spaces
3. Builds model with those hyperparameters
4. Returns compiled model

Keras Tuner will call this function many times
with different hyperparameter combinations.
"""

def build_tunable_model(hp, input_shape):
    """
    Build LSTM model with tunable hyperparameters.

    Args:
        hp: Keras Tuner HyperParameters object
        input_shape: (sequence_length, n_features)

    Returns:
        Compiled Keras model
    """
    model = Sequential()
    model.add(Input(shape=input_shape))

    # Tune LSTM Layer 1 units
    units_1 = hp.Choice('units_layer1', values=[64, 128, 256])
    model.add(LSTM(units_1, return_sequences=True))

    # Tune dropout rate (shared across all layers)
    dropout_rate = hp.Choice('dropout_rate', values=[0.1, 0.2, 0.3])
    model.add(Dropout(dropout_rate))

    # Tune LSTM Layer 2 units
    units_2 = hp.Choice('units_layer2', values=[32, 64, 128])
    model.add(LSTM(units_2, return_sequences=True))
    model.add(Dropout(dropout_rate))

    # Tune LSTM Layer 3 units
    units_3 = hp.Choice('units_layer3', values=[16, 32, 64])
    model.add(LSTM(units_3, return_sequences=False))
    model.add(Dropout(dropout_rate))

    # Dense layers
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='linear'))

    # Tune learning rate
    learning_rate = hp.Choice('learning_rate', values=[0.0001, 0.001, 0.01])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )

    return model

print("\n✅ Tunable model builder created!")
print("\n📊 Hyperparameter Search Space:")
print("   • units_layer1:   [64, 128, 256]    → 3 options")
print("   • units_layer2:   [32, 64, 128]     → 3 options")
print("   • units_layer3:   [16, 32, 64]      → 3 options")
print("   • dropout_rate:   [0.1, 0.2, 0.3]   → 3 options")
print("   • learning_rate:  [0.0001, 0.001, 0.01] → 3 options")
print("\n   Total combinations: 3 × 3 × 3 × 3 × 3 = 243 possible models")
print("   We'll try 20 (Bayesian search finds best quickly!)")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Building Tunable LSTM Model

✅ Tunable model builder created!

📊 Hyperparameter Search Space:
   • units_layer1:   [64, 128, 256]    → 3 options
   • units_layer2:   [32, 64, 128]     → 3 options
   • units_layer3:   [16, 32, 64]      → 3 options
   • dropout_rate:   [0.1, 0.2, 0.3]   → 3 options
   • learning_rate:  [0.0001, 0.001, 0.01] → 3 options

   Total combinations: 3 × 3 × 3 × 3 × 3 = 243 possible models
   We'll try 20 (Bayesian search finds best quickly!)

✅ Exercise 1.2 Complete!


In [6]:
# ==================================================
# EXERCISE 1.3: PREPARE DATA FOR TUNING
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.3: Preparing Data for Hyperparameter Tuning")
print("=" * 80)

"""
📖 THEORY: Data Preparation for Tuning

We'll tune on ONE stock (NVDA) to save time:
- Tuning on all 5 stocks would take 5× longer
- Best hyperparameters from NVDA work well for others
- We can fine-tune per-stock later if needed

Using the baseline 60-day sequence from Day 87.
"""

print("\n⏱️ Preparing NVDA data for tuning...")

SEQUENCE_LENGTH = 60  # Same as Day 87

def create_sequences(data, target_idx, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i, :])
        y.append(data[i, target_idx])
    return np.array(X), np.array(y)

# Prepare NVDA data
ticker = 'NVDA'
df = stock_data_clean[ticker]

# Scale
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df.values)

# Create sequences
X, y = create_sequences(scaled, target_idx, SEQUENCE_LENGTH)

# Split
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

total = len(X)
train_end = int(total * TRAIN_RATIO)
val_end = int(total * (TRAIN_RATIO + VAL_RATIO))

X_train_tune = X[:train_end]
y_train_tune = y[:train_end]
X_val_tune = X[train_end:val_end]
y_val_tune = y[train_end:val_end]

print(f"\n✅ Data prepared for {ticker}:")
print(f"   Train: {X_train_tune.shape}")
print(f"   Val:   {X_val_tune.shape}")
print(f"   Input shape: {X_train_tune.shape[1:]}")

# Store input shape for model builder
input_shape_tune = (X_train_tune.shape[1], X_train_tune.shape[2])

print("\n✅ Exercise 1.3 Complete!")
print("=" * 80)


EXERCISE 1.3: Preparing Data for Hyperparameter Tuning

⏱️ Preparing NVDA data for tuning...

✅ Data prepared for NVDA:
   Train: (802, 60, 34)
   Val:   (172, 60, 34)
   Input shape: (60, 34)

✅ Exercise 1.3 Complete!


In [7]:
# ==================================================
# EXERCISE 1.4: RUN HYPERPARAMETER SEARCH
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.4: Running Hyperparameter Search")
print("=" * 80)

"""
📖 THEORY: Bayesian Optimization

Bayesian Optimization is SMART:
1. Tries a few random combinations
2. Builds a probabilistic model of performance
3. Predicts which combinations will be best
4. Focuses search on promising regions
5. Much faster than grid search!

We use BayesianOptimization tuner from Keras Tuner.
"""

print("\n⚙️ Setting up Bayesian Optimization tuner...")

# Create tuner
tuner = kt.BayesianOptimization(
    lambda hp: build_tunable_model(hp, input_shape_tune),
    objective='val_loss',
    max_trials=20,  # Try 20 different combinations
    executions_per_trial=1,
    directory='tuning_results',
    project_name='stock_lstm_tuning',
    overwrite=True
)

print(f"✅ Tuner created!")
print(f"   Algorithm: Bayesian Optimization")
print(f"   Max trials: 20")
print(f"   Objective: Minimize val_loss")

print("\n⏱️ Starting hyperparameter search...")
print("   This will take 40-60 minutes (GPU accelerated)")
print("   Progress will be shown below:\n")

# Run search
tuner.search(
    X_train_tune,
    y_train_tune,
    validation_data=(X_val_tune, y_val_tune),
    epochs=50,  # Max epochs per trial (EarlyStopping will stop earlier)
    batch_size=32,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ],
    verbose=1
)

print("\n✅ Hyperparameter search complete!")
print("=" * 80)

Trial 20 Complete [00h 00m 15s]
val_loss: 0.003497172612696886

Best val_loss So Far: 0.00219285162165761
Total elapsed time: 00h 03m 34s

✅ Hyperparameter search complete!


In [8]:
# ==================================================
# EXERCISE 1.5: ANALYZE BEST HYPERPARAMETERS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.5: Best Hyperparameters Found")
print("=" * 80)

# Get best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n🏆 BEST HYPERPARAMETERS:")
print("=" * 60)
print(f"   LSTM Layer 1 Units:  {best_hp.get('units_layer1')}")
print(f"   LSTM Layer 2 Units:  {best_hp.get('units_layer2')}")
print(f"   LSTM Layer 3 Units:  {best_hp.get('units_layer3')}")
print(f"   Dropout Rate:        {best_hp.get('dropout_rate')}")
print(f"   Learning Rate:       {best_hp.get('learning_rate')}")
print("=" * 60)

# Get best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate best model
train_loss, train_mae = best_model.evaluate(X_train_tune, y_train_tune, verbose=0)
val_loss, val_mae = best_model.evaluate(X_val_tune, y_val_tune, verbose=0)

print("\n📊 Best Model Performance:")
print(f"   Train Loss: {train_loss:.6f}")
print(f"   Train MAE:  {train_mae:.6f}")
print(f"   Val Loss:   {val_loss:.6f}")
print(f"   Val MAE:    {val_mae:.6f}")

# Compare with Day 87 baseline (if you remember the value)
print("\n💡 Comparison with Day 87 Baseline:")
print("   Day 87 used: 128-64-32 units, 0.2 dropout, 0.001 LR")
print("   Check if tuned model improved val_loss!")

# Show top 5 trials
print("\n📊 Top 5 Trials:")
print(f"\n{'Trial':<8} {'Val Loss':>12} {'Units (L1-L2-L3)':>20} {'Dropout':>10} {'LR':>10}")
print("─" * 80)

for i, trial in enumerate(tuner.oracle.get_best_trials(num_trials=5)):
    hp_values = trial.hyperparameters.values
    units_str = f"{hp_values['units_layer1']}-{hp_values['units_layer2']}-{hp_values['units_layer3']}"

    print(f"{i+1:<8} {trial.score:>12.6f} {units_str:>20} {hp_values['dropout_rate']:>10.1f} {hp_values['learning_rate']:>10.4f}")

print("\n✅ Exercise 1.5 Complete!")
print("=" * 80)


EXERCISE 1.5: Best Hyperparameters Found

🏆 BEST HYPERPARAMETERS:
   LSTM Layer 1 Units:  256
   LSTM Layer 2 Units:  64
   LSTM Layer 3 Units:  64
   Dropout Rate:        0.1
   Learning Rate:       0.01

📊 Best Model Performance:
   Train Loss: 0.000286
   Train MAE:  0.012831
   Val Loss:   0.002193
   Val MAE:    0.040809

💡 Comparison with Day 87 Baseline:
   Day 87 used: 128-64-32 units, 0.2 dropout, 0.001 LR
   Check if tuned model improved val_loss!

📊 Top 5 Trials:

Trial        Val Loss     Units (L1-L2-L3)    Dropout         LR
────────────────────────────────────────────────────────────────────────────────
1            0.002193            256-64-64        0.1     0.0100
2            0.002407             64-32-32        0.1     0.0001
3            0.002425            128-32-32        0.2     0.0010
4            0.002820            128-64-64        0.1     0.0010
5            0.002961             64-64-32        0.1     0.0010

✅ Exercise 1.5 Complete!


In [9]:
# ==================================================
# EXERCISE 1.6: EXPERIMENT WITH SEQUENCE LENGTHS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.6: Testing Different Sequence Lengths")
print("=" * 80)

"""
📖 THEORY: Sequence Length Impact

Sequence length = how many days of history to use:
- 30 days:  Less noise, faster training, may miss patterns
- 60 days:  Good balance (our baseline)
- 90 days:  More context, more noise, slower
- 120 days: Maximum context, very slow, may overfit

We test each to find the sweet spot!
"""

print("\n⏱️ Testing sequence lengths: 30, 60, 90, 120 days...")

sequence_results = {}

for seq_len in [30, 60, 90, 120]:
    print(f"\n{'─'*60}")
    print(f"Testing sequence length: {seq_len} days")
    print(f"{'─'*60}")

    # Create sequences with this length
    X_seq, y_seq = create_sequences(scaled, target_idx, seq_len)

    # Split
    total = len(X_seq)
    train_end = int(total * 0.70)
    val_end = int(total * 0.85)

    X_train_seq = X_seq[:train_end]
    y_train_seq = y_seq[:train_end]
    X_val_seq = X_seq[train_end:val_end]
    y_val_seq = y_seq[train_end:val_end]

    # Build model with best hyperparameters
    input_shape_seq = (X_train_seq.shape[1], X_train_seq.shape[2])
    model_seq = build_tunable_model(best_hp, input_shape_seq)

    # Train
    history = model_seq.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=50,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=0
    )

    best_val_loss = min(history.history['val_loss'])
    sequence_results[seq_len] = best_val_loss

    print(f"   ✅ Best val_loss: {best_val_loss:.6f}")
    print(f"   Epochs trained: {len(history.history['loss'])}")

# Find best sequence length
best_seq_len = min(sequence_results, key=sequence_results.get)

print(f"\n{'='*60}")
print("📊 Sequence Length Results:")
print(f"{'='*60}")
for seq_len, val_loss in sorted(sequence_results.items()):
    marker = "🏆" if seq_len == best_seq_len else "  "
    print(f"{marker} {seq_len} days: val_loss = {val_loss:.6f}")

print(f"\n🏆 Best sequence length: {best_seq_len} days")
print(f"   Val loss: {sequence_results[best_seq_len]:.6f}")

print("\n✅ Exercise 1.6 Complete!")
print("=" * 80)


EXERCISE 1.6: Testing Different Sequence Lengths

⏱️ Testing sequence lengths: 30, 60, 90, 120 days...

────────────────────────────────────────────────────────────
Testing sequence length: 30 days
────────────────────────────────────────────────────────────
   ✅ Best val_loss: 0.003339
   Epochs trained: 11

────────────────────────────────────────────────────────────
Testing sequence length: 60 days
────────────────────────────────────────────────────────────
   ✅ Best val_loss: 0.003812
   Epochs trained: 23

────────────────────────────────────────────────────────────
Testing sequence length: 90 days
────────────────────────────────────────────────────────────
   ✅ Best val_loss: 0.001413
   Epochs trained: 28

────────────────────────────────────────────────────────────
Testing sequence length: 120 days
────────────────────────────────────────────────────────────
   ✅ Best val_loss: 0.003907
   Epochs trained: 10

📊 Sequence Length Results:
   30 days: val_loss = 0.003339
   60 d

In [ ]:
print("\n" + "=" * 80)
print("🔄 PART 2: WALK-FORWARD VALIDATION")
print("=" * 80)


🔄 PART 2: WALK-FORWARD VALIDATION


In [ ]:
# ==================================================
# EXERCISE 2.1: WALK-FORWARD VALIDATION THEORY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Walk-Forward Validation Concept")
print("=" * 80)

"""
📖 THEORY: Walk-Forward Validation

Walk-forward validation simulates REAL TRADING:

Traditional Train/Test Split:
   Train: [==================]
   Test:                      [=====]
   Problem: Only tests ONCE on one time period

Walk-Forward Validation:
   Period 1: Train [==========]  Test [==]
   Period 2: Train [============]  Test [==]
   Period 3: Train [==============]  Test [==]
   Period 4: Train [================]  Test [==]
   ...

How it works:
1. Start with initial training window
2. Predict next N days (test window)
3. Add test window to training data
4. Retrain model on expanded window
5. Predict next N days
6. Repeat until end of data

Why it's better:
✅ Tests model on multiple time periods
✅ Simulates real trading (expand data over time)
✅ More robust performance estimate
✅ Detects if model degrades over time

Example:
- Initial train: 2020-2023 (3 years)
- Test window: 30 days
- Period 1: Train on 2020-2023 → Test Jan 2024
- Period 2: Train on 2020-Jan 2024 → Test Feb 2024
- Period 3: Train on 2020-Feb 2024 → Test Mar 2024
- Continue through all of 2024-2025
"""

print("\n📊 Walk-Forward Strategy:")
print("   • Initial training: First 70% of data")
print("   • Test window: 30 days at a time")
print("   • Retrain: After each 30-day period")
print("   • Total periods: ~10-15 periods")

print("\n💡 Why Retrain?")
print("   • Market conditions change over time")
print("   • Model learns from recent data")
print("   • Adapts to new patterns")
print("   • More realistic than static model")

print("\n⚠️ Computational Cost:")
print("   • Must retrain model 10-15 times")
print("   • Each training takes 2-5 minutes")
print("   • Total: 20-75 minutes on GPU")
print("   • Worth it for robust validation!")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Walk-Forward Validation Concept

📊 Walk-Forward Strategy:
   • Initial training: First 70% of data
   • Test window: 30 days at a time
   • Retrain: After each 30-day period
   • Total periods: ~10-15 periods

💡 Why Retrain?
   • Market conditions change over time
   • Model learns from recent data
   • Adapts to new patterns
   • More realistic than static model

⚠️ Computational Cost:
   • Must retrain model 10-15 times
   • Each training takes 2-5 minutes
   • Total: 20-75 minutes on GPU
   • Worth it for robust validation!

✅ Exercise 2.1 Complete!


In [ ]:
# ==================================================
# EXERCISE 2.2: IMPLEMENT WALK-FORWARD VALIDATION
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Implementing Walk-Forward Validation")
print("=" * 80)

"""
📖 THEORY: Implementation Strategy

We'll implement walk-forward on NVDA with:
- Best hyperparameters from tuning
- Best sequence length
- 30-day test windows
- Expanding training window
"""

print("\n⏱️ Setting up walk-forward validation...")

# Use best hyperparameters and sequence length
BEST_SEQ_LEN = best_seq_len  # From Exercise 1.6
TEST_WINDOW = 30  # Predict 30 days at a time

print(f"\n📊 Walk-Forward Configuration:")
print(f"   Sequence Length: {BEST_SEQ_LEN} days")
print(f"   Test Window: {TEST_WINDOW} days")
print(f"   Stock: NVDA")

# Prepare data with best sequence length
ticker = 'NVDA'
df = stock_data_clean[ticker]
scaler_wf = MinMaxScaler(feature_range=(0, 1))
scaled_wf = scaler_wf.fit_transform(df.values)

X_wf, y_wf = create_sequences(scaled_wf, target_idx, BEST_SEQ_LEN)

print(f"   Total samples: {len(X_wf)}")

# Calculate walk-forward splits
INITIAL_TRAIN_SIZE = int(len(X_wf) * 0.70)
remaining = len(X_wf) - INITIAL_TRAIN_SIZE
n_periods = remaining // TEST_WINDOW

print(f"   Initial training samples: {INITIAL_TRAIN_SIZE}")
print(f"   Number of test periods: {n_periods}")
print(f"   Total retrains: {n_periods}")

print("\n⏱️ Running walk-forward validation...")
print("   This will take 20-40 minutes...\n")

wf_predictions = []
wf_actuals = []
wf_periods = []

for period in range(n_periods):
    print(f"{'─'*60}")
    print(f"Period {period + 1}/{n_periods}")
    print(f"{'─'*60}")

    # Define training window (expanding)
    train_end = INITIAL_TRAIN_SIZE + (period * TEST_WINDOW)
    test_start = train_end
    test_end = test_start + TEST_WINDOW

    # Check if we have enough data
    if test_end > len(X_wf):
        test_end = len(X_wf)

    # Get data
    X_train_wf = X_wf[:train_end]
    y_train_wf = y_wf[:train_end]
    X_test_wf = X_wf[test_start:test_end]
    y_test_wf = y_wf[test_start:test_end]

    if len(X_test_wf) == 0:
        break

    print(f"   Train size: {len(X_train_wf)}")
    print(f"   Test size:  {len(X_test_wf)}")

    # Build and train model with best hyperparameters
    input_shape_wf = (X_train_wf.shape[1], X_train_wf.shape[2])
    model_wf = build_tunable_model(best_hp, input_shape_wf)

    # Train (silent)
    model_wf.fit(
        X_train_wf, y_train_wf,
        epochs=50,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)],
        verbose=0
    )

    # Predict
    y_pred_wf = model_wf.predict(X_test_wf, verbose=0).flatten()

    # Store results
    wf_predictions.extend(y_pred_wf)
    wf_actuals.extend(y_test_wf)
    wf_periods.extend([period + 1] * len(y_pred_wf))

    # Calculate period metrics
    rmse_period = np.sqrt(mean_squared_error(y_test_wf, y_pred_wf))
    print(f"   Period RMSE: {rmse_period:.6f}")
    print(f"   ✅ Done\n")

# Convert to arrays
wf_predictions = np.array(wf_predictions)
wf_actuals = np.array(wf_actuals)
wf_periods = np.array(wf_periods)

print(f"{'='*60}")
print("✅ Walk-Forward Validation Complete!")
print(f"{'='*60}")
print(f"   Total predictions: {len(wf_predictions)}")
print(f"   Total periods: {n_periods}")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Implementing Walk-Forward Validation

⏱️ Setting up walk-forward validation...

📊 Walk-Forward Configuration:
   Sequence Length: 90 days
   Test Window: 30 days
   Stock: NVDA
   Total samples: 1116
   Initial training samples: 781
   Number of test periods: 11
   Total retrains: 11

⏱️ Running walk-forward validation...
   This will take 20-40 minutes...

────────────────────────────────────────────────────────────
Period 1/11
────────────────────────────────────────────────────────────
   Train size: 781
   Test size:  30
   Period RMSE: 0.049693
   ✅ Done

────────────────────────────────────────────────────────────
Period 2/11
────────────────────────────────────────────────────────────
   Train size: 811
   Test size:  30
   Period RMSE: 0.055938
   ✅ Done

────────────────────────────────────────────────────────────
Period 3/11
────────────────────────────────────────────────────────────
   Train size: 841
   Test size:  30
   Period RMSE: 0.063903
   ✅ Done

────

   Period RMSE: 0.069856
   ✅ Done

────────────────────────────────────────────────────────────
Period 6/11
────────────────────────────────────────────────────────────
   Train size: 931
   Test size:  30


   Period RMSE: 0.027942
   ✅ Done

────────────────────────────────────────────────────────────
Period 7/11
────────────────────────────────────────────────────────────
   Train size: 961
   Test size:  30
   Period RMSE: 0.100000
   ✅ Done

────────────────────────────────────────────────────────────
Period 8/11
────────────────────────────────────────────────────────────
   Train size: 991
   Test size:  30
   Period RMSE: 0.095203
   ✅ Done

────────────────────────────────────────────────────────────
Period 9/11
────────────────────────────────────────────────────────────
   Train size: 1021
   Test size:  30
   Period RMSE: 0.129044
   ✅ Done

────────────────────────────────────────────────────────────
Period 10/11
────────────────────────────────────────────────────────────
   Train size: 1051
   Test size:  30
   Period RMSE: 0.069545
   ✅ Done

────────────────────────────────────────────────────────────
Period 11/11
───────────────────────────────────────────────────────────

In [ ]:
# ==================================================
# EXERCISE 2.3: INVERSE TRANSFORM & CALCULATE METRICS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.3: Walk-Forward Performance Metrics")
print("=" * 80)

print("\n⏱️ Inverse transforming predictions to real prices...")

# Inverse transform predictions and actuals
n_features = len(feature_columns)

# Predictions
dummy_pred = np.zeros((len(wf_predictions), n_features))
dummy_pred[:, target_idx] = wf_predictions
wf_pred_prices = scaler_wf.inverse_transform(dummy_pred)[:, target_idx]

# Actuals
dummy_actual = np.zeros((len(wf_actuals), n_features))
dummy_actual[:, target_idx] = wf_actuals
wf_actual_prices = scaler_wf.inverse_transform(dummy_actual)[:, target_idx]

print("✅ Inverse transform complete!")

# Calculate overall metrics
wf_rmse = np.sqrt(mean_squared_error(wf_actual_prices, wf_pred_prices))
wf_mae = mean_absolute_error(wf_actual_prices, wf_pred_prices)
wf_mape = np.mean(np.abs((wf_actual_prices - wf_pred_prices) / wf_actual_prices)) * 100

print("\n📊 Walk-Forward Overall Metrics:")
print(f"   RMSE: ${wf_rmse:.2f}")
print(f"   MAE:  ${wf_mae:.2f}")
print(f"   MAPE: {wf_mape:.2f}%")

# Calculate per-period metrics
print("\n📊 Performance by Period:")
print(f"\n{'Period':>8} {'Samples':>10} {'RMSE':>12} {'MAE':>12} {'MAPE':>10}")
print("─" * 60)

for period in range(1, n_periods + 1):
    mask = wf_periods == period
    if mask.sum() == 0:
        continue

    period_actual = wf_actual_prices[mask]
    period_pred = wf_pred_prices[mask]

    period_rmse = np.sqrt(mean_squared_error(period_actual, period_pred))
    period_mae = mean_absolute_error(period_actual, period_pred)
    period_mape = np.mean(np.abs((period_actual - period_pred) / period_actual)) * 100

    print(f"{period:>8} {mask.sum():>10} ${period_rmse:>11.2f} ${period_mae:>11.2f} {period_mape:>9.2f}%")

print("\n✅ Exercise 2.3 Complete!")
print("=" * 80)


EXERCISE 2.3: Walk-Forward Performance Metrics

⏱️ Inverse transforming predictions to real prices...


NameError: name 'feature_columns' is not defined